In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# Load data
test_data=pd.read_csv('/kaggle/input/titanic/test.csv')
train_data=pd.read_csv('/kaggle/input/titanic/train.csv')
# =========================
# FEATURE ENGINEERING
# =========================

# Family size
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1

# Is alone
train_data['IsAlone'] = np.where(train_data['FamilySize'] == 1, 1, 0)
test_data['IsAlone'] = np.where(test_data['FamilySize'] == 1, 1, 0)

# Extract title from Name
train_data['Title'] = train_data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test_data['Title'] = test_data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Simplify rare titles
rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
train_data['Title'] = train_data['Title'].replace(rare_titles,'Rare')
test_data['Title'] = test_data['Title'].replace(rare_titles,'Rare')

# Age bin
train_data['AgeBin'] = pd.cut(train_data['Age'], bins=[0,12,20,40,60,80], labels=[0,1,2,3,4])
test_data['AgeBin'] = pd.cut(test_data['Age'], bins=[0,12,20,40,60,80], labels=[0,1,2,3,4])

# Fare bin
train_data['FareBin'] = pd.qcut(train_data['Fare'], 4, labels=[0,1,2,3])
test_data['FareBin'] = pd.qcut(test_data['Fare'], 4, labels=[0,1,2,3])

# =========================
# FEATURES & TARGET
# =========================

features = ['Pclass','Sex','Title','FamilySize','IsAlone','AgeBin','FareBin']
X_train = pd.get_dummies(train_data[features])
y_train = train_data['Survived']

X_test = pd.get_dummies(test_data[features])

# Make sure train and test have same columns
X_test = X_test.reindex(columns = X_train.columns, fill_value=0)

# =========================
# BALANCED CLASS WEIGHT
# =========================
neg = sum(y_train == 0)
pos = sum(y_train == 1)
scale_pos_weight = neg / pos

# =========================
# XGB + GridSearchCV
# =========================

xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

param_grid = {
    'max_depth': [3,4,5],
    'learning_rate': [0.1,0.05],
    'n_estimators': [100,200]
}

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='f1',   # optimize for F1 of survivors
    cv=3,
    verbose=1
)

grid.fit(X_train, y_train)
print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# =========================
# PREDICTION
# =========================
predictions = grid.predict(X_test)

# If you have y_test (from train/test split), evaluate:
# X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
# y_pred_val = grid.predict(X_val)
# print(classification_report(y_val, y_pred_val))

# =========================
# SAVE SUBMISSION
# =========================
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission_improved.csv', index=False)
print("Submission saved!")
